# Session 09 Practice: Path Optimization & Velocity Profiling
**Course:** Motion Planning for Autonomous Vehicles (Planning P2)  
**Week 5 | Session 9**  

---

### 🎯 Learning Objectives
1. Fit smooth B-splines to coarse waypoints and analytically evaluate path curvature $\kappa(s)$.
2. Enforce the **lateral acceleration limit**: $v_{\max}(s) = \sqrt{\frac{a_{\text{lat,max}}}{|\kappa(s)|}}$.
3. Compute forward and backward dynamic integration passes for **acceleration ($a_{\text{accel}}$) and braking ($a_{\text{decel}}$)** limits.
4. Visualize smooth spatial trajectories, curvature profiles, and dynamic velocity envelopes with **Plotly**.

---

### 📐 Mathematical Formulation

#### Analytical Path Curvature
$$\kappa(s) = \frac{\dot{x}(s) \ddot{y}(s) - \dot{y}(s) \ddot{x}(s)}{(\dot{x}(s)^2 + \dot{y}(s)^2)^{3/2}}$$

#### Dynamic Velocity Profiling Algorithm
1. **Curvature Bound:** $v_{\text{curv}}(s_i) = \min\left(v_{\text{cruise}}, \sqrt{\frac{a_{\text{lat,max}}}{|\kappa(s_i)|}}\right)$ with $v(s_N) = 0$.
2. **Backward Pass (Braking):**
   $$v_i = \min\left(v_i, \sqrt{v_{i+1}^2 + 2 a_{\text{decel}} \Delta s_i}\right)$$
3. **Forward Pass (Acceleration):**
   $$v_{i+1} = \min\left(v_{i+1}, \sqrt{v_i^2 + 2 a_{\text{accel}} \Delta s_i}\right)$$


In [ ]:
import math
import numpy as np
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.interpolate import splprep, splev
import plotly.io as pio
pio.templates.default = "plotly_white"


## 1. Spline Path Fitting & Analytical Curvature

In [ ]:
def generate_curved_path_waypoints() -> tuple[np.ndarray, np.ndarray]:
    x = [0.0, 30.0, 60.0, 80.0, 90.0, 95.0, 95.0, 95.0]
    y = [0.0, 0.0, 0.0, 10.0, 30.0, 50.0, 80.0, 110.0]
    return np.array(x), np.array(y)

def fit_smooth_spline_path(x_pts: np.ndarray, y_pts: np.ndarray, num_samples: int = 150):
    tck, u = splprep([x_pts, y_pts], s=0, k=3)
    u_new = np.linspace(0, 1, num_samples)
    x_smooth, y_smooth = splev(u_new, tck)
    
    # Derivatives
    dx, dy = splev(u_new, tck, der=1)
    ddx, ddy = splev(u_new, tck, der=2)
    
    # Path curvature
    curvature = (dx * ddy - dy * ddx) / ((dx**2 + dy**2)**1.5)
    
    # Cumulative arc length s
    ds = np.sqrt(np.diff(x_smooth)**2 + np.diff(y_smooth)**2)
    s = np.concatenate(([0.0], np.cumsum(ds)))
    
    return s, x_smooth, y_smooth, curvature


## 2. Dynamic Forward-Backward Pass Velocity Profiler

In [ ]:
def compute_velocity_profile(s: np.ndarray, curvature: np.ndarray,
                             v_cruise: float = 12.0,
                             a_lat_max: float = 2.5,
                             a_accel_max: float = 1.5,
                             a_decel_max: float = 2.0,
                             v_init: float = 0.0) -> np.ndarray:
    N = len(s)
    v_max = np.zeros(N)
    
    for i in range(N):
        k = abs(curvature[i])
        if k > 1e-4:
            v_curv = math.sqrt(a_lat_max / k)
        else:
            v_curv = v_cruise
        v_max[i] = min(v_cruise, v_curv)
        
    v_max[-1] = 0.0 # Stop at goal
    
    # --- Backward Pass ---
    v_profile = np.copy(v_max)
    for i in range(N - 2, -1, -1):
        ds = s[i+1] - s[i]
        v_allowed = math.sqrt(v_profile[i+1]**2 + 2 * a_decel_max * ds)
        v_profile[i] = min(v_profile[i], v_allowed)
        
    # --- Forward Pass ---
    v_profile[0] = v_init
    for i in range(0, N - 1):
        ds = s[i+1] - s[i]
        v_allowed = math.sqrt(v_profile[i]**2 + 2 * a_accel_max * ds)
        v_profile[i+1] = min(v_profile[i+1], v_allowed)
        
    return v_profile

x_coarse, y_coarse = generate_curved_path_waypoints()
s, x_smooth, y_smooth, curvature = fit_smooth_spline_path(x_coarse, y_coarse)
v_prof = compute_velocity_profile(s, curvature, v_cruise=12.0, a_lat_max=2.5, a_accel_max=1.5, a_decel_max=2.0)
print(f"Velocity profile computed across {len(s)} arc-length waypoints.")


## 3. Interactive Multi-Panel Optimization Dashboard

In [ ]:
def plot_optimization_dashboard(s, x, y, curvature, v_profile, a_lat_max=2.5):
    a_lat = (v_profile ** 2) * np.abs(curvature)
    
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            "Optimized Smooth Path (Global X vs Y)",
            "Path Curvature κ(s)",
            "Dynamic Velocity Profile v(s)",
            "Lateral Acceleration a_lat(s) vs Comfort Limit"
        ),
        vertical_spacing=0.12,
        horizontal_spacing=0.10
    )
    
    # 1. 2D Path
    fig.add_trace(go.Scatter(x=x, y=y, mode="lines", name="Smooth Path",
                             line=dict(color="#1f77b4", width=3.5)), row=1, col=1)
    fig.add_trace(go.Scatter(x=x_coarse, y=y_coarse, mode="markers", name="Coarse Waypoints",
                             marker=dict(size=8, color="red")), row=1, col=1)
    
    # 2. Curvature
    fig.add_trace(go.Scatter(x=s, y=curvature, mode="lines", name="κ(s)",
                             line=dict(color="#ff7f0e", width=2.5)), row=1, col=2)
    
    # 3. Velocity
    fig.add_trace(go.Scatter(x=s, y=v_profile, mode="lines", name="Velocity v(s)",
                             line=dict(color="#2ca02c", width=3)), row=2, col=1)
    
    # 4. Lateral Acceleration
    fig.add_trace(go.Scatter(x=s, y=a_lat, mode="lines", name="a_lat (m/s²)",
                             line=dict(color="#9467bd", width=2.5)), row=2, col=2)
    fig.add_hline(y=a_lat_max, line_dash="dash", line_color="crimson",
                  annotation_text="Comfort Limit (2.5 m/s²)", row=2, col=2)
    
    fig.update_layout(title="Trajectory Optimization & Dynamic Velocity Profiling Dashboard", height=720, showlegend=False)
    fig.show()

plot_optimization_dashboard(s, x_smooth, y_smooth, curvature, v_prof)


## 4. Verification Tests

In [ ]:
# Verify constraint compliance
a_lat_actual = (v_prof ** 2) * np.abs(curvature)
max_alat = float(np.max(a_lat_actual))
print(f"Maximum lateral acceleration encountered: {max_alat:.3f} m/s² (Limit: 2.5 m/s²)")
assert max_alat <= 2.5 + 1e-3, f"Lateral acceleration limit violated: {max_alat} > 2.5!"

# Verify goal stop
assert math.isclose(v_prof[-1], 0.0, abs_tol=1e-3), "Vehicle must come to a complete stop at path end!"

print("✅ Velocity profiling constraint checks passed successfully!")
